In [2]:
import os, glob
import pandas as pd
import numpy as np
from google.colab import files

archivos = sorted([
    f for f in glob.glob("/content/*.xls*")
    if os.path.basename(f).lower() not in ["hurto.xlsx", "hurto.xls"]
])

salida = "/content/diagnostico_bases.txt"
lineas = []

for ruta in archivos:
    nombre = os.path.basename(ruta)
    try:
        df = pd.read_excel(ruta, engine="calamine")
    except Exception:
        df = pd.read_excel(ruta, engine="openpyxl")

    lineas.append("=" * 80)
    lineas.append(f"BASE: {nombre} | Filas: {len(df):,} | Columnas: {df.shape[1]} | Duplicados: {df.duplicated().sum():,}")
    lineas.append("=" * 80)

    resumen = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos_%": (df.isna().mean() * 100).round(2),
        "unicos": df.nunique(dropna=True),
        "ejemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
    })
    lineas.append("\n[1] ESTRUCTURA:\n" + resumen.to_string())

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        lineas.append("\n[2] NUMÉRICAS:\n" + df[num_cols].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']].round(2).to_string())

    lineas.append("\n[3] PRIMERAS 3 FILAS:\n" + df.head(3).to_string() + "\n")

with open(salida, "w", encoding="utf-8") as f:
    f.write("\n".join(lineas))

print(f"✅ Listo. Archivo guardado en {salida}")
files.download(salida)

✅ Listo. Archivo guardado en /content/diagnostico_bases.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import os
import pandas as pd
import numpy as np

# 1. Crear estructura de carpetas del repositorio
BASE_DIR = "/content/repositorio_observatorio"
for sub in ["data/clean", "docs", "notebooks"]:
    os.makedirs(f"{BASE_DIR}/{sub}", exist_ok=True)

archivos_config = {
    "Amenazas.xlsx": "amenazas",
    "Delitos Sexuales.xlsx": "delitos_sexuales",
    "HomicidioIntencional.xlsx": "homicidio_intencional",
    "HurtoComercio.xlsx": "hurto_comercio",
    "Secuestro.xlsx": "secuestro",
    "Terrorismo.xlsx": "terrorismo",
}

def limpiar_base(ruta, nombre_delito):
    try:
        df = pd.read_excel(ruta, engine="calamine")
    except Exception:
        df = pd.read_excel(ruta, engine="openpyxl")

    # Quitar espacios extra en nombres de columnas y pasar a minúsculas
    df.columns = [str(c).strip().upper() for c in df.columns]

    # Renombrar columnas equivalentes a un esquema único
    mapa_cols = {
        "ARMA MEDIO": "arma_medio", "ARMAS MEDIOS": "arma_medio", "ARMAS_MEDIOS": "arma_medio",
        "DEPARTAMENTO": "departamento",
        "MUNICIPIO": "municipio",
        "FECHA HECHO": "fecha", "FECHA": "fecha",
        "GENERO": "genero",
        "*AGRUPA_EDAD_PERSONA": "grupo_edad", "*AGRUPA EDAD PERSONA*": "grupo_edad",
        "CODIGO DANE": "codigo_dane", "CODIGO_DANE": "codigo_dane",
        "CANTIDAD": "cantidad"
    }
    df = df.rename(columns={k: v for k, v in mapa_cols.items() if k in df.columns})

    # Asegurar columnas opcionales (HurtoComercio y Terrorismo no tienen género/edad)
    for col_opt in ["genero", "grupo_edad"]:
        if col_opt not in df.columns:
            df[col_opt] = "NO APLICA"
        else:
            df[col_opt] = df[col_opt].fillna("NO REPORTA").astype(str).str.strip().str.upper()

    df["arma_medio"] = df["arma_medio"].fillna("NO REPORTA").astype(str).str.strip().str.upper()
    df["departamento"] = df["departamento"].astype(str).str.strip().str.upper()
    df["municipio"] = df["municipio"].astype(str).str.strip().str.upper()

    # Parseo robusto de fechas (soporta YYYYMMDD y YYYY-MM-DD)
    if nombre_delito == "homicidio_intencional":
        df["fecha"] = pd.to_datetime(df["fecha"].astype(str).str.strip(), format="%Y%m%d", errors="coerce")
    else:
        df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    df["anio"] = df["fecha"].dt.year
    # Filtrar al rango válido del proyecto (2013 - 2025) eliminando errores como 1905
    df = df[(df["anio"] >= 2013) & (df["anio"] <= 2025)].copy()
    df["anio"] = df["anio"].astype(int)

    # Normalizar código DANE a 8 dígitos (ej: 5001000 -> 05001000)
    df["codigo_dane"] = pd.to_numeric(df["codigo_dane"], errors="coerce").fillna(0).astype(int).astype(str).str.zfill(8)
    df["cod_depto"] = df["codigo_dane"].str[:2]
    df["cod_mpio"] = df["codigo_dane"].str[:5]

    # Separar Bogotá D.C. (cod_depto == '11') de Cundinamarca para precisión territorial
    df.loc[df["cod_depto"] == "11", "departamento"] = "BOGOTÁ D.C."

    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0).astype(int)
    df["tipo_delito"] = nombre_delito

    cols_finales = ["cod_depto", "departamento", "cod_mpio", "municipio", "anio", "fecha",
                    "tipo_delito", "arma_medio", "genero", "grupo_edad", "cantidad"]
    return df[cols_finales]

# 2. Procesar y unificar las 6 bases
dfs = []
for archivo, delito in archivos_config.items():
    ruta = f"/content/{archivo}"
    if os.path.exists(ruta):
        print(f"Limpiando {archivo}...", end=" ")
        d = limpiar_base(ruta, delito)
        dfs.append(d)
        print(f"OK -> {len(d):,} filas válidas (2013-2025)")

df_unificado = pd.concat(dfs, ignore_index=True)

# Guardar base unificada en formato Parquet (liviano y rápido para el repositorio)
df_unificado.to_parquet(f"{BASE_DIR}/data/clean/eventos_seguridad_unificado.parquet", index=False)

# 3. Construir el Panel Departamental-Anual (base estructurada para análisis y modelos ML)
panel_delitos = df_unificado.pivot_table(
    index=["cod_depto", "departamento", "anio"],
    columns="tipo_delito",
    values="cantidad",
    aggfunc="sum",
    fill_value=0
).reset_index()

# Variables estructurales / contextuales por Departamento y Año
df_unificado["es_arma_fuego"] = df_unificado["arma_medio"].str.contains("FUEGO", na=False) * df_unificado["cantidad"]
df_unificado["es_arma_blanca"] = df_unificado["arma_medio"].str.contains("BLANCA|CORTOPUNZANTE", na=False) * df_unificado["cantidad"]
df_unificado["es_femenino"] = (df_unificado["genero"] == "FEMENINO") * df_unificado["cantidad"]
df_unificado["es_menores"] = df_unificado["grupo_edad"].str.contains("MENOR|ADOLESCENTE", na=False) * df_unificado["cantidad"]

ctx = df_unificado.groupby(["cod_depto", "departamento", "anio"]).agg(
    total_delitos=("cantidad", "sum"),
    municipios_afectados=("cod_mpio", "nunique"),
    casos_arma_fuego=("es_arma_fuego", "sum"),
    casos_arma_blanca=("es_arma_blanca", "sum"),
    casos_victima_femenina=("es_femenino", "sum"),
    casos_victima_menores=("es_menores", "sum"),
).reset_index()

ctx["pct_arma_fuego"] = (ctx["casos_arma_fuego"] / ctx["total_delitos"] * 100).round(2)
ctx["pct_arma_blanca"] = (ctx["casos_arma_blanca"] / ctx["total_delitos"] * 100).round(2)
ctx["pct_victima_femenina"] = (ctx["casos_victima_femenina"] / ctx["total_delitos"] * 100).round(2)
ctx["pct_victima_menores"] = (ctx["casos_victima_menores"] / ctx["total_delitos"] * 100).round(2)

# Unir todo en el Panel Maestro
panel = panel_delitos.merge(ctx, on=["cod_depto", "departamento", "anio"], how="left")
panel = panel.sort_values(["cod_depto", "anio"]).reset_index(drop=True)

# Guardar Panel Maestro en CSV (legible en GitHub y Excel)
ruta_panel = f"{BASE_DIR}/data/clean/panel_departamental_seguridad.csv"
panel.to_csv(ruta_panel, index=False, encoding="utf-8-sig")

print("\n✅ PASO 1 COMPLETADO")
print(f"Panel Maestro guardado en: {ruta_panel}")
print(f"Dimensiones del panel: {panel.shape[0]} filas x {panel.shape[1]} columnas")
print(f"Departamentos ({panel['departamento'].nunique()}):", sorted(panel["departamento"].unique())[:10], "...")
print(f"Años cubiertos:", sorted(panel["anio"].unique()))
display(panel.head(5))

Limpiando Amenazas.xlsx... OK -> 401,939 filas válidas (2013-2025)
Limpiando Delitos Sexuales.xlsx... OK -> 293,479 filas válidas (2013-2025)
Limpiando HomicidioIntencional.xlsx... OK -> 84,430 filas válidas (2013-2025)
Limpiando HurtoComercio.xlsx... OK -> 332,278 filas válidas (2013-2025)
Limpiando Secuestro.xlsx... OK -> 2,621 filas válidas (2013-2025)
Limpiando Terrorismo.xlsx... OK -> 2,774 filas válidas (2013-2025)

✅ PASO 1 COMPLETADO
Panel Maestro guardado en: /content/repositorio_observatorio/data/clean/panel_departamental_seguridad.csv
Dimensiones del panel: 364 filas x 19 columnas
Departamentos (33): ['AMAZONAS', 'ANTIOQUIA', 'ARAUCA', 'ATLÁNTICO', 'BOGOTÁ D.C.', 'BOLÍVAR', 'BOYACÁ', 'CALDAS', 'CAQUETÁ', 'CASANARE'] ...
Años cubiertos: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,cod_depto,departamento,anio,amenazas,delitos_sexuales,homicidio_intencional,hurto_comercio,secuestro,terrorismo,total_delitos,municipios_afectados,casos_arma_fuego,casos_arma_blanca,casos_victima_femenina,casos_victima_menores,pct_arma_fuego,pct_arma_blanca,pct_victima_femenina,pct_victima_menores
0,00,GUAINÍA,2017,0,1,0,0,0,0,1,1,0,0,1,1,0.00,0.00,100.00,100.00
1,05,ANTIOQUIA,2015,3109,2917,1890,3366,26,55,11363,125,2315,657,3973,2120,20.37,5.78,34.96,18.66
2,05,ANTIOQUIA,2016,2591,3096,1861,3442,37,35,11062,125,2222,647,3890,2463,20.09,5.85,35.17,22.27
3,05,ANTIOQUIA,2017,2585,3674,1919,8495,17,15,16705,125,3558,747,4363,2903,21.30,4.47,26.12,17.38
4,05,ANTIOQUIA,2018,4805,4756,2439,18730,28,34,30792,124,5869,1654,6247,3428,19.06,5.37,20.29,11.13


In [4]:
import pandas as pd
import numpy as np

BASE_DIR = "/content/repositorio_observatorio"

# --- FIX RÁPIDO: Rescatar fechas mixtas en HomicidioIntencional ---
df_h = pd.read_excel("/content/HomicidioIntencional.xlsx", engine="calamine")
df_h.columns = [str(c).strip().upper() for c in df_h.columns]
df_h = df_h.rename(columns={
    "ARMAS MEDIOS": "arma_medio", "DEPARTAMENTO": "departamento", "MUNICIPIO": "municipio",
    "FECHA HECHO": "fecha", "GENERO": "genero", "*AGRUPA EDAD PERSONA*": "grupo_edad",
    "CODIGO DANE": "codigo_dane", "CANTIDAD": "cantidad"
})
# Intenta YYYYMMDD y donde falle usa parseo general
f1 = pd.to_datetime(df_h["fecha"].astype(str).str.strip(), format="%Y%m%d", errors="coerce")
f2 = pd.to_datetime(df_h["fecha"], errors="coerce")
df_h["fecha"] = f1.fillna(f2)
df_h["anio"] = df_h["fecha"].dt.year
df_h = df_h[(df_h["anio"] >= 2015) & (df_h["anio"] <= 2025)].copy()
df_h["anio"] = df_h["anio"].astype(int)
df_h["codigo_dane"] = pd.to_numeric(df_h["codigo_dane"], errors="coerce").fillna(0).astype(int).astype(str).str.zfill(8)
df_h["cod_depto"] = df_h["codigo_dane"].str[:2]
df_h["cod_mpio"] = df_h["codigo_dane"].str[:5]
df_h["departamento"] = df_h["departamento"].astype(str).str.strip().str.upper()
df_h.loc[df_h["cod_depto"] == "11", "departamento"] = "BOGOTÁ D.C."
df_h["municipio"] = df_h["municipio"].astype(str).str.strip().str.upper()
df_h["arma_medio"] = df_h["arma_medio"].fillna("NO REPORTA").astype(str).str.strip().str.upper()
df_h["genero"] = df_h["genero"].fillna("NO REPORTA").astype(str).str.strip().str.upper()
df_h["grupo_edad"] = df_h["grupo_edad"].fillna("NO REPORTA").astype(str).str.strip().str.upper()
df_h["cantidad"] = pd.to_numeric(df_h["cantidad"], errors="coerce").fillna(0).astype(int)
df_h["tipo_delito"] = "homicidio_intencional"

cols = ["cod_depto", "departamento", "cod_mpio", "municipio", "anio", "fecha",
        "tipo_delito", "arma_medio", "genero", "grupo_edad", "cantidad"]

# Reemplazar homicidios en el unificado y reconstruir panel limpio
df_uni = pd.read_parquet(f"{BASE_DIR}/data/clean/eventos_seguridad_unificado.parquet")
df_uni = pd.concat([df_uni[df_uni["tipo_delito"] != "homicidio_intencional"], df_h[cols]], ignore_index=True)
df_uni = df_uni[df_uni["cod_depto"] != "00"].copy()
df_uni.to_parquet(f"{BASE_DIR}/data/clean/eventos_seguridad_unificado.parquet", index=False)

print(f"✅ Homicidios rescatados: {len(df_h):,} filas válidas (2015-2025)")

panel_delitos = df_uni.pivot_table(
    index=["cod_depto", "departamento", "anio"], columns="tipo_delito",
    values="cantidad", aggfunc="sum", fill_value=0
).reset_index()

df_uni["es_arma_fuego"] = df_uni["arma_medio"].str.contains("FUEGO", na=False) * df_uni["cantidad"]
df_uni["es_arma_blanca"] = df_uni["arma_medio"].str.contains("BLANCA|CORTOPUNZANTE", na=False) * df_uni["cantidad"]
df_uni["es_femenino"] = (df_uni["genero"] == "FEMENINO") * df_uni["cantidad"]
df_uni["es_menores"] = df_uni["grupo_edad"].str.contains("MENOR|ADOLESCENTE", na=False) * df_uni["cantidad"]

ctx = df_uni.groupby(["cod_depto", "departamento", "anio"]).agg(
    total_delitos=("cantidad", "sum"),
    municipios_afectados=("cod_mpio", "nunique"),
    casos_arma_fuego=("es_arma_fuego", "sum"),
    casos_arma_blanca=("es_arma_blanca", "sum"),
    casos_victima_femenina=("es_femenino", "sum"),
    casos_victima_menores=("es_menores", "sum"),
).reset_index()

for num, den, col in [
    ("casos_arma_fuego", "total_delitos", "pct_arma_fuego"),
    ("casos_arma_blanca", "total_delitos", "pct_arma_blanca"),
    ("casos_victima_femenina", "total_delitos", "pct_victima_femenina"),
    ("casos_victima_menores", "total_delitos", "pct_victima_menores"),
]:
    ctx[col] = (ctx[num] / ctx[den] * 100).round(2)

panel = panel_delitos.merge(ctx, on=["cod_depto", "departamento", "anio"], how="left")
panel = panel.sort_values(["cod_depto", "anio"]).reset_index(drop=True)
panel.to_csv(f"{BASE_DIR}/data/clean/panel_departamental_seguridad.csv", index=False, encoding="utf-8-sig")

# --- PASO 2: ESTUDIO ESTADÍSTICO PARA VARIABLES DEPENDIENTES E INDEPENDIENTES ---
vars_num = [
    "homicidio_intencional", "amenazas", "delitos_sexuales", "hurto_comercio",
    "secuestro", "terrorismo", "municipios_afectados",
    "pct_arma_fuego", "pct_arma_blanca", "pct_victima_femenina", "pct_victima_menores"
]

print("\n📊 1. ESTADÍSTICOS DESCRIPTIVOS DEL PANEL (33 deptos x 11 años):")
print(panel[vars_num].describe().T[["mean", "std", "min", "50%", "max"]].round(2).to_string())

print("\n🔗 2. CORRELACIÓN DE SPEARMAN CON HOMICIDIO INTENCIONAL (posible Y1) Y HURTO COMERCIO (posible Y2):")
corr = panel[vars_num].corr(method="spearman")
print(corr[["homicidio_intencional", "hurto_comercio"]].sort_values("homicidio_intencional", ascending=False).round(3).to_string())

✅ Homicidios rescatados: 114,475 filas válidas (2015-2025)

📊 1. ESTADÍSTICOS DESCRIPTIVOS DEL PANEL (33 deptos x 11 años):
                          mean      std    min      50%       max
homicidio_intencional   376.44   549.41   0.00   216.00   3684.00
amenazas               1693.08  2062.26   7.00  1154.00  13833.00
delitos_sexuales        917.00  1267.52  11.00   611.00   9667.00
hurto_comercio         1483.32  3122.20   0.00   626.00  37258.00
secuestro                 8.00    16.47   0.00     3.00    243.00
terrorismo                7.91    19.14   0.00     1.00    177.00
municipios_afectados     33.21    33.21   1.00    26.00    125.00
pct_arma_fuego           12.50     6.84   0.00    12.52     38.37
pct_arma_blanca           5.39     3.15   0.00     5.03     20.25
pct_victima_femenina     37.19     7.26  18.51    36.89     61.76
pct_victima_menores      16.72     6.24   3.86    15.45     42.86

🔗 2. CORRELACIÓN DE SPEARMAN CON HOMICIDIO INTENCIONAL (posible Y1) Y HURTO COMERCI

In [5]:
!pip install -q python-calamine openpyxl seaborn matplotlib

import os
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# ==============================================================================
# 1. ESTRUCTURA DEL REPOSITORIO
# ==============================================================================
BASE_DIR = "/content/repositorio_observatorio"
for sub in ["data/clean", "docs", "notebooks"]:
    os.makedirs(f"{BASE_DIR}/{sub}", exist_ok=True)

archivos_config = {
    "Amenazas.xlsx": "amenazas",
    "Delitos Sexuales.xlsx": "delitos_sexuales",
    "HomicidioIntencional.xlsx": "homicidio_intencional",
    "HurtoComercio.xlsx": "hurto_comercio",
    "Secuestro.xlsx": "secuestro",
    "Terrorismo.xlsx": "terrorismo",
}

# ==============================================================================
# 2. LIMPIEZA Y UNIFICACIÓN (ETL)
# ==============================================================================
def limpiar_base(ruta, nombre_delito):
    try:
        df = pd.read_excel(ruta, engine="calamine")
    except Exception:
        df = pd.read_excel(ruta, engine="openpyxl")

    df.columns = [str(c).strip().upper() for c in df.columns]

    mapa_cols = {
        "ARMA MEDIO": "arma_medio", "ARMAS MEDIOS": "arma_medio", "ARMAS_MEDIOS": "arma_medio",
        "DEPARTAMENTO": "departamento",
        "MUNICIPIO": "municipio",
        "FECHA HECHO": "fecha", "FECHA": "fecha",
        "GENERO": "genero",
        "*AGRUPA_EDAD_PERSONA": "grupo_edad", "*AGRUPA EDAD PERSONA*": "grupo_edad",
        "CODIGO DANE": "codigo_dane", "CODIGO_DANE": "codigo_dane",
        "CANTIDAD": "cantidad"
    }
    df = df.rename(columns={k: v for k, v in mapa_cols.items() if k in df.columns})

    for col_opt in ["genero", "grupo_edad"]:
        if col_opt not in df.columns:
            df[col_opt] = "NO APLICA"
        else:
            df[col_opt] = df[col_opt].fillna("NO REPORTA").astype(str).str.strip().str.upper()

    df["arma_medio"] = df["arma_medio"].fillna("NO REPORTA").astype(str).str.strip().str.upper()
    df["departamento"] = df["departamento"].astype(str).str.strip().str.upper()
    df["municipio"] = df["municipio"].astype(str).str.strip().str.upper()

    # Parseo robusto para fechas mixtas (YYYYMMDD y YYYY-MM-DD)
    if nombre_delito == "homicidio_intencional":
        f1 = pd.to_datetime(df["fecha"].astype(str).str.strip(), format="%Y%m%d", errors="coerce")
        f2 = pd.to_datetime(df["fecha"], errors="coerce")
        df["fecha"] = f1.fillna(f2)
    else:
        df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    df["anio"] = df["fecha"].dt.year
    df = df[(df["anio"] >= 2015) & (df["anio"] <= 2025)].copy()
    df["anio"] = df["anio"].astype(int)

    df["codigo_dane"] = pd.to_numeric(df["codigo_dane"], errors="coerce").fillna(0).astype(int).astype(str).str.zfill(8)
    df["cod_depto"] = df["codigo_dane"].str[:2]
    df["cod_mpio"] = df["codigo_dane"].str[:5]
    df = df[df["cod_depto"] != "00"].copy()

    # Separar Bogotá D.C. de Cundinamarca
    df.loc[df["cod_depto"] == "11", "departamento"] = "BOGOTÁ D.C."

    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0).astype(int)
    df["tipo_delito"] = nombre_delito

    cols_finales = ["cod_depto", "departamento", "cod_mpio", "municipio", "anio", "fecha",
                    "tipo_delito", "arma_medio", "genero", "grupo_edad", "cantidad"]
    return df[cols_finales]

dfs = []
for archivo, delito in archivos_config.items():
    ruta = f"/content/{archivo}"
    if os.path.exists(ruta):
        print(f"⏳ Limpiando {archivo}...", end=" ", flush=True)
        d = limpiar_base(ruta, delito)
        dfs.append(d)
        print(f"OK -> {len(d):,} filas")
    else:
        print(f"⚠️ No se encontró: {archivo}")

df_uni = pd.concat(dfs, ignore_index=True)
df_uni.to_parquet(f"{BASE_DIR}/data/clean/eventos_seguridad_unificado.parquet", index=False)

# ==============================================================================
# 3. CONSTRUCCIÓN DEL PANEL DEPARTAMENTAL-ANUAL (33 deptos x 11 años)
# ==============================================================================
panel_delitos = df_uni.pivot_table(
    index=["cod_depto", "departamento", "anio"],
    columns="tipo_delito",
    values="cantidad",
    aggfunc="sum",
    fill_value=0
).reset_index()

df_uni["es_arma_fuego"] = df_uni["arma_medio"].str.contains("FUEGO", na=False) * df_uni["cantidad"]
df_uni["es_arma_blanca"] = df_uni["arma_medio"].str.contains("BLANCA|CORTOPUNZANTE", na=False) * df_uni["cantidad"]
df_uni["es_femenino"] = (df_uni["genero"] == "FEMENINO") * df_uni["cantidad"]
df_uni["es_menores"] = df_uni["grupo_edad"].str.contains("MENOR|ADOLESCENTE", na=False) * df_uni["cantidad"]

ctx = df_uni.groupby(["cod_depto", "departamento", "anio"]).agg(
    total_delitos=("cantidad", "sum"),
    municipios_afectados=("cod_mpio", "nunique"),
    casos_arma_fuego=("es_arma_fuego", "sum"),
    casos_arma_blanca=("es_arma_blanca", "sum"),
    casos_victima_femenina=("es_femenino", "sum"),
    casos_victima_menores=("es_menores", "sum"),
).reset_index()

for num, den, col in [
    ("casos_arma_fuego", "total_delitos", "pct_arma_fuego"),
    ("casos_arma_blanca", "total_delitos", "pct_arma_blanca"),
    ("casos_victima_femenina", "total_delitos", "pct_victima_femenina"),
    ("casos_victima_menores", "total_delitos", "pct_victima_menores"),
]:
    ctx[col] = (ctx[num] / ctx[den] * 100).round(2)

panel = panel_delitos.merge(ctx, on=["cod_depto", "departamento", "anio"], how="left")
panel = panel.sort_values(["cod_depto", "anio"]).reset_index(drop=True)
panel.to_csv(f"{BASE_DIR}/data/clean/panel_departamental_seguridad.csv", index=False, encoding="utf-8-sig")

# ==============================================================================
# 4. ESTADÍSTICOS Y CORRELACIÓN DE SPEARMAN
# ==============================================================================
vars_num = [
    "homicidio_intencional", "hurto_comercio", "amenazas", "delitos_sexuales",
    "secuestro", "terrorismo", "municipios_afectados",
    "pct_arma_fuego", "pct_arma_blanca", "pct_victima_femenina", "pct_victima_menores"
]

print("\n📊 ESTADÍSTICOS DESCRIPTIVOS:")
print(panel[vars_num].describe().T[["mean", "std", "min", "50%", "max"]].round(2).to_string())

print("\n🔗 CORRELACIÓN DE SPEARMAN:")
corr = panel[vars_num].corr(method="spearman")
print(corr[["homicidio_intencional", "hurto_comercio"]].sort_values("homicidio_intencional", ascending=False).round(3).to_string())

# ==============================================================================
# 5. DOCUMENTACIÓN (README + DICCIONARIO DE DATOS)
# ==============================================================================
readme_md = """# Observatorio de Seguridad Ciudadana — Colombia (2015–2025)

## 1. Descripción del Proyecto
Repositorio estructurado con datos oficiales de la Policía Nacional de Colombia (2015–2025), limpiados, estandarizados y consolidados en un panel departamental-anual (33 entidades territoriales × 11 años) para el análisis estadístico y modelado predictivo de la seguridad ciudadana.

## 2. Estructura del Repositorio

    repositorio_observatorio/
    ├── data/
    │   └── clean/
    │       ├── eventos_seguridad_unificado.parquet   # Base transaccional unificada (6 delitos)
    │       └── panel_departamental_seguridad.csv     # Panel maestro (33 deptos x 11 años)
    ├── docs/
    │   ├── diccionario_datos.md                      # Definición y rol estadístico de variables
    │   ├── heatmap_correlaciones.png                 # Matriz de correlación de Spearman
    │   └── tendencia_delitos_2015_2025.png           # Evolución anual de delitos
    ├── notebooks/                                    # Notebooks de ETL y análisis exploratorio
    └── README.md

## 3. Decisiones de Limpieza (ETL)
1. **Exclusión controlada:** Se excluyó `Hurto.xlsx` por corrupción de archivo; se integraron las 6 bases íntegras (`Amenazas`, `Delitos Sexuales`, `HomicidioIntencional`, `HurtoComercio`, `Secuestro`, `Terrorismo`).
2. **Estandarización de esquema:** Unificación de nombres de columnas con espacios residuales y nomenclaturas heterogéneas.
3. **Normalización de fechas:** Rescate de formatos mixtos (`YYYYMMDD` y `YYYY-MM-DD`) en `HomicidioIntencional` y filtrado de registros anómalos (ej. año 1905 en `Terrorismo`) al periodo 2015–2025.
4. **Precisión territorial (DANE):** Normalización del código DANE a 8 dígitos y separación de **Bogotá D.C.** (`cod_depto = 11`) respecto a Cundinamarca.
5. **Conservación de registros:** No se eliminaron filas idénticas porque en registros policiales representan denuncias independientes ocurridas el mismo día en el mismo municipio.

## 4. Selección Estadística de Variables
Debido a la asimetría positiva de los conteos delictivos, se utilizó correlación no paramétrica de **Spearman**:
- **Variables Dependientes (Y):** `homicidio_intencional` (violencia letal) y `hurto_comercio` (delito patrimonial).
- **Variables Independientes (X):** `amenazas`, `delitos_sexuales`, `secuestro`, `terrorismo`, `municipios_afectados`, `pct_arma_fuego`, `pct_arma_blanca`, `pct_victima_femenina`, `pct_victima_menores`.
- **Hallazgo clave:** `pct_arma_fuego` presenta una correlación fuerte con `homicidio_intencional` (ρ = 0.539) frente a una correlación débil con `hurto_comercio` (ρ = 0.112).
"""

diccionario_md = """# Diccionario de Datos — `panel_departamental_seguridad.csv`

| Variable | Tipo | Rol | Descripción |
| :--- | :--- | :--- | :--- |
| `cod_depto` | string (2 dígitos) | Identificador | Código DANE departamental estandarizado (`11` = Bogotá D.C.). |
| `departamento` | string | Identificador | Nombre del departamento o Distrito Capital (33 territorios). |
| `anio` | int | Identificador / X | Año del hecho delictivo (2015–2025). |
| `homicidio_intencional` | int | **Dependiente (Y1)** | Total anual de homicidios intencionales registrados. |
| `hurto_comercio` | int | **Dependiente (Y2)** | Total anual de hurtos a comercio registrados. |
| `amenazas` | int | Independiente (X) | Total anual de denuncias por amenazas. |
| `delitos_sexuales` | int | Independiente (X) | Total anual de delitos sexuales registrados. |
| `secuestro` | int | Independiente (X) | Total anual de casos de secuestro. |
| `terrorismo` | int | Independiente (X) | Total anual de actos de terrorismo. |
| `total_delitos` | int | Agregado | Suma total de eventos de las 6 bases en el departamento y año. |
| `municipios_afectados` | int | Independiente (X) | Número de municipios distintos con al menos un delito en el año. |
| `pct_arma_fuego` | float (%) | Independiente (X) | Porcentaje de delitos cometidos con arma de fuego. |
| `pct_arma_blanca` | float (%) | Independiente (X) | Porcentaje de delitos cometidos con arma blanca/cortopunzante. |
| `pct_victima_femenina` | float (%) | Independiente (X) | Porcentaje de casos con víctima de género femenino. |
| `pct_victima_menores` | float (%) | Independiente (X) | Porcentaje de casos con víctimas menores o adolescentes. |
"""

with open(f"{BASE_DIR}/README.md", "w", encoding="utf-8") as f:
    f.write(readme_md)

with open(f"{BASE_DIR}/docs/diccionario_datos.md", "w", encoding="utf-8") as f:
    f.write(diccionario_md)

# ==============================================================================
# 6. GRÁFICAS Y DESCARGA DEL REPOSITORIO (.ZIP)
# ==============================================================================
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlación de Spearman — Variables de Seguridad (2015–2025)")
plt.tight_layout()
plt.savefig(f"{BASE_DIR}/docs/heatmap_correlaciones.png", dpi=150)
plt.close()

tend = panel.groupby("anio")[["homicidio_intencional", "hurto_comercio", "amenazas", "delitos_sexuales"]].sum()
tend.plot(figsize=(10, 5), marker="o")
plt.title("Evolución Anual de Delitos en Colombia (2015–2025)")
plt.ylabel("Número de casos")
plt.xlabel("Año")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{BASE_DIR}/docs/tendencia_delitos_2015_2025.png", dpi=150)
plt.close()

zip_path = shutil.make_archive("/content/repositorio_observatorio", "zip", BASE_DIR)
print(f"\n✅ TODO LISTO. Descargando: {zip_path}")
files.download(zip_path)

⏳ Limpiando Amenazas.xlsx... OK -> 401,939 filas
⏳ Limpiando Delitos Sexuales.xlsx... OK -> 293,478 filas
⏳ Limpiando HomicidioIntencional.xlsx... OK -> 114,475 filas
⏳ Limpiando HurtoComercio.xlsx... OK -> 332,278 filas
⏳ Limpiando Secuestro.xlsx... OK -> 2,621 filas
⏳ Limpiando Terrorismo.xlsx... OK -> 2,774 filas

📊 ESTADÍSTICOS DESCRIPTIVOS:
                          mean      std    min      50%       max
homicidio_intencional   376.44   549.41   0.00   216.00   3684.00
hurto_comercio         1483.32  3122.20   0.00   626.00  37258.00
amenazas               1693.08  2062.26   7.00  1154.00  13833.00
delitos_sexuales        917.00  1267.52  11.00   611.00   9667.00
secuestro                 8.00    16.47   0.00     3.00    243.00
terrorismo                7.91    19.14   0.00     1.00    177.00
municipios_afectados     33.21    33.21   1.00    26.00    125.00
pct_arma_fuego           12.50     6.84   0.00    12.52     38.37
pct_arma_blanca           5.39     3.15   0.00     5.03   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>